In [54]:
import mlflow
import pandas as pd
import os
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_class_weight # No genera datos sinteticos como SMOTE, sino que hace un diccionario asignandolé mas peso a clases minoritarias, para que el modelo se esfuerce mas por predecirlas bien.
from mlflow.keras import log_model as keras_log_model
from mlflow.sklearn import log_model as sklearn_log_model
from mlflow import set_tag

In [55]:
df = pd.read_csv("../data/wdbc_clean.csv")

# 1. Detectar Entorno (antes de todo MLflow)
uri = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")  # Remoto si existe, sino local

# 2. Setear Tracking URI
mlflow.set_tracking_uri(uri)

# 3. Definir Experimento
mlflow.set_experiment("Proyecto_RNN_Clasificacion")  # Crea o selecciona experimento

<Experiment: artifact_location='file:///c:/Users/320258215/Desktop/IA-ML/repaso_sklearn/notebooks_train/mlruns/2', creation_time=1772634183427, experiment_id='2', last_update_time=1772634183427, lifecycle_stage='active', name='Proyecto_RNN_Clasificacion', tags={}>

In [56]:
from mlflow import log_input


def preprocess_data(clean_df, params_spliter, target_col="target", irrelevant_cols=None):

    
    if irrelevant_cols is not None:
        clean_df = clean_df.drop(columns=irrelevant_cols)
    
    clean_df[target_col] = clean_df[target_col].map({'M': 1, 'B': 0})

    # Split
    train_df, test_df = train_test_split(clean_df, test_size=params_spliter["test_size"], random_state=params_spliter["random_state"])
    
    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col]

    data_train = mlflow.data.from_pandas(train_df, targets=target_col, name="Train_Set")
    data_test = mlflow.data.from_pandas(test_df, targets=target_col, name="Test_Set")
    log_input(data_train, context="training")
    log_input(data_test, context="testing")

    numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns

    scaler = StandardScaler()
    
    # No hace falta hacer concatenaciones
    preprocessor = ColumnTransformer(
        transformers=[
            ("scaler", scaler, numerical_cols),
            ("encoder", OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
            ]
    )

    # mlflow.sklearn.log_model() IMPORTANTE, VIENE DE SCIKIT LEARN, ME FALLABA EL CODIGO PORQUE LO HABÍA IMPORTADO DESDE KERAS
    sklearn_log_model(preprocessor, artifact_path="preprocessor", registered_model_name="Preprocessor_RNN_Clasificacion")

    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled = preprocessor.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, preprocessor


In [57]:
def balanced_classes(X_train, y_train):
    
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    class_weights_dict = dict(zip(np.unique(y_train), class_weights))

    return class_weights_dict

In [58]:


def build_model(input_dim, class_weights, adam, params_compile):

    
    model = Sequential()
    model.add(Dense(64, activation='relu', input_dim=input_dim))
    model.add(Dropout(0.5))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))  # Para clasificación binaria

    model.compile(**params_compile)
    
    mlflow.log_params({
        "input_dim" : input_dim,
        "model_architecture": "Dense(64) -> Dropout(0.5) -> Dense(32) -> Dropout(0.5) -> Dense(1)",
        "optimizer": adam,
        "loss_function": params_compile["loss"],
        "metrics": params_compile["metrics"],
        "class_weights": class_weights,
    })

    
    return model

In [ ]:
def train_model (X_train, y_train, X_test, y_test, rnn_model, params_train):
    
    history = rnn_model.fit(**params_train)
    
     # el [-1] es para tomar el último valor de la lista, que corresponde a la última época.
    mlflow.log_metrics({
        "final_loss": history.history['loss'][-1], 
        "final_accuracy": history.history['accuracy'][-1],
        "final_val_loss": history.history['val_loss'][-1],
        "final_val_accuracy": history.history['val_accuracy'][-1]
    })

    # Cargar el mejor modelo guardado por el callback
    best_model = tf.keras.models.load_model("./models/best_model_RNN.h5")

    # mlflow.keras.log_model
    keras_log_model(best_model, name="best_model", registered_model_name="RNN_Clasificacion_Best") 

    return rnn_model, history

In [60]:

with mlflow.start_run():

    if not os.path.exists('./models'): os.makedirs('./models')

    set_tag("model_type", "RNN")
    set_tag("dataset", "WDBC")
    set_tag("framework", "Keras")
    set_tag("user", "Lucas Palú")
    set_tag("env", "test")

    
    # =========================================================================================================
    # Preprocesamiento de datos y parametros
    # =========================================================================================================
    target_col = "target"
    irrelevant_cols = None
    params_spliter = {
        "test_size": 0.2,
        "random_state": 42
    }

    X_train, X_test, y_train, y_test, preprocessor = preprocess_data(df, params_spliter, target_col, irrelevant_cols)


    # =========================================================================================================
    # Calculo de class weights para manejar el desbalanceo de clases
    # =========================================================================================================    
    class_weights_dict = balanced_classes(X_train, y_train)


    # =========================================================================================================
    # Construcción del modelo y parametros de compilación
    # =========================================================================================================
    input_dim = X_train.shape[1]
    adam = Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    )
    params_compile = {
        "optimizer": adam,
        "loss": 'binary_crossentropy',
        "metrics": ['accuracy', 'AUC', 'Precision', 'Recall', 'TruePositives', 'TrueNegatives', 'FalsePositives', 'FalseNegatives'],
        "weighted_metrics": ['AUC']
    }

    rnn_model = build_model(input_dim, class_weights_dict, adam, params_compile)


    # ========================================================================================================
    # Parametros para el entrenamiento y entrenamiento
    # ========================================================================================================
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('./models/best_model_RNN.h5', monitor='val_loss', save_best_only=True), 
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ]
    params_train = {
        "x": X_train,
        "y": y_train,
        "epochs": 50,
        "batch_size": 10, 
        "verbose": 1,
        "validation_split": 0.2, 
        "class_weight": class_weights_dict,
        "callbacks": callbacks
    }
    
    trained_model, history = train_model(X_train, y_train, X_test, y_test, rnn_model, params_train) # history es un diccionario con la evolución de las métricas durante el entrenamiento, se puede loggear en MLflow para luego analizarlo.

    mlflow.keras.log_model(trained_model, artifact_path="model", registered_model_name="RNN_Cancer_Model") # Esto es para loggear el modelo final entrenado, no el mejor modelo guardado por el callback, que se loggea aparte. Es importante loggear ambos para poder comparar el rendimiento del modelo final con el mejor modelo encontrado durante el entrenamiento.

# Importar set_tag para etiquetar el experimento con información relevante


c:\Users\320258215\AppData\Local\anaconda4\envs\heart-ml\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/04 15:14:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 15:14:59 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_functi

Epoch 1/50
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - AUC: 0.6077 - FalseNegatives: 19.6944 - FalsePositives: 55.8889 - Precision: 0.4573 - Recall: 0.7087 - TrueNegatives: 60.6944 - TruePositives: 48.7222 - accuracy: 0.5509 - loss: 0.7845 - weighted_AUC: 0.6077

37/37 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - AUC: 0.7471 - FalseNegatives: 36.0000 - FalsePositives: 87.0000 - Precision: 0.5348 - Recall: 0.7353 - TrueNegatives: 141.0000 - TruePositives: 100.0000 - accuracy: 0.6621 - loss: 0.6373 - weighted_AUC: 0.7471 - val_AUC: 0.9522 - val_FalseNegatives: 3.0000 - val_FalsePositives: 9.0000 - val_Precision: 0.7692 - val_Recall: 0.9091 - val_TrueNegatives: 49.0000 - val_TruePositives: 30.0000 - val_accuracy: 0.8681 - val_loss: 0.3796 - val_weighted_AUC: 0.9522 - learning_rate: 0.0010
Epoch 2/50
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - AUC: 0.8729 - FalseNegatives: 9.8611 - FalsePositives: 23.9722 - Precision: 0.7030 - Recall: 0.8488 - TrueNegatives: 91.7778 - TruePositives: 59.3889 - accuracy: 0.8019 - loss: 0.4565 - weighted_AUC: 0.8729

37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - AUC: 0.9073 - FalseNegatives: 18.0000 - FalsePositives: 44.0000 - Precision: 0.7284 - Recall: 0.8676 - TrueNegatives: 184.0000 - TruePositives: 118.0000 - accuracy: 0.8297 - loss: 0.4011 - weighted_AUC: 0.9073 - val_AUC: 0.9796 - val_FalseNegatives: 2.0000 - val_FalsePositives: 4.0000 - val_Precision: 0.8857 - val_Recall: 0.9394 - val_TrueNegatives: 54.0000 - val_TruePositives: 31.0000 - val_accuracy: 0.9341 - val_loss: 0.2405 - val_weighted_AUC: 0.9796 - learning_rate: 0.0010
Epoch 3/50
35/37 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - AUC: 0.9546 - FalseNegatives: 5.5143 - FalsePositives: 14.0000 - Precision: 0.8084 - Recall: 0.9053 - TrueNegatives: 97.4000 - TruePositives: 63.0857 - accuracy: 0.8790 - loss: 0.2867 - weighted_AUC: 0.9546

37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - AUC: 0.9681 - FalseNegatives: 13.0000 - FalsePositives: 24.0000 - Precision: 0.8367 - Recall: 0.9044 - TrueNegatives: 204.0000 - TruePositives: 123.0000 - accuracy: 0.8984 - loss: 0.2624 - weighted_AUC: 0.9681 - val_AUC: 0.9869 - val_FalseNegatives: 1.0000 - val_FalsePositives: 3.0000 - val_Precision: 0.9143 - val_Recall: 0.9697 - val_TrueNegatives: 55.0000 - val_TruePositives: 32.0000 - val_accuracy: 0.9560 - val_loss: 0.1759 - val_weighted_AUC: 0.9869 - learning_rate: 0.0010
Epoch 4/50
34/37 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - AUC: 0.9613 - FalseNegatives: 2.8235 - FalsePositives: 8.9118 - Precision: 0.8752 - Recall: 0.9460 - TrueNegatives: 101.2941 - TruePositives: 61.9706 - accuracy: 0.9286 - loss: 0.2659 - weighted_AUC: 0.9613

...

37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - AUC: 0.9797 - FalseNegatives: 6.0000 - FalsePositives: 15.0000 - Precision: 0.8966 - Recall: 0.9559 - TrueNegatives: 213.0000 - TruePositives: 130.0000 - accuracy: 0.9423 - loss: 0.1793 - weighted_AUC: 0.9797 - val_AUC: 0.9937 - val_FalseNegatives: 1.0000 - val_FalsePositives: 2.0000 - val_Precision: 0.9412 - val_Recall: 0.9697 - val_TrueNegatives: 56.0000 - val_TruePositives: 32.0000 - val_accuracy: 0.9670 - val_loss: 0.1093 - val_weighted_AUC: 0.9937 - learning_rate: 0.0010
Epoch 8/50
35/37 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.9925 - FalseNegatives: 2.6286 - FalsePositives: 2.7714 - Precision: 0.9638 - Recall: 0.9502 - TrueNegatives: 108.8286 - TruePositives: 65.7714 - accuracy: 0.9685 - loss: 0.1154 - weighted_AUC: 0.9925

37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - AUC: 0.9913 - FalseNegatives: 4.0000 - FalsePositives: 9.0000 - Precision: 0.9362 - Recall: 0.9706 - TrueNegatives: 219.0000 - TruePositives: 132.0000 - accuracy: 0.9643 - loss: 0.1228 - weighted_AUC: 0.9913 - val_AUC: 0.9935 - val_FalseNegatives: 1.0000 - val_FalsePositives: 3.0000 - val_Precision: 0.9143 - val_Recall: 0.9697 - val_TrueNegatives: 55.0000 - val_TruePositives: 32.0000 - val_accuracy: 0.9560 - val_loss: 0.1071 - val_weighted_AUC: 0.9935 - learning_rate: 0.0010
Epoch 9/50
33/37 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - AUC: 0.9898 - FalseNegatives: 3.5152 - FalsePositives: 3.9697 - Precision: 0.9059 - Recall: 0.9079 - TrueNegatives: 106.9091 - TruePositives: 55.6061 - accuracy: 0.9429 - loss: 0.1230 - weighted_AUC: 0.9898

37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - AUC: 0.9938 - FalseNegatives: 5.0000 - FalsePositives: 7.0000 - Precision: 0.9493 - Recall: 0.9632 - TrueNegatives: 221.0000 - TruePositives: 131.0000 - accuracy: 0.9670 - loss: 0.1055 - weighted_AUC: 0.9938 - val_AUC: 0.9924 - val_FalseNegatives: 1.0000 - val_FalsePositives: 3.0000 - val_Precision: 0.9143 - val_Recall: 0.9697 - val_TrueNegatives: 55.0000 - val_TruePositives: 32.0000 - val_accuracy: 0.9560 - val_loss: 0.1067 - val_weighted_AUC: 0.9924 - learning_rate: 0.0010
Epoch 10/50
35/37 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - AUC: 0.9940 - FalseNegatives: 1.8857 - FalsePositives: 2.1429 - Precision: 0.9704 - Recall: 0.9726 - TrueNegatives: 111.3714 - TruePositives: 64.6000 - accuracy: 0.9792 - loss: 0.0944 - weighted_AUC: 0.9940

37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - AUC: 0.9920 - FalseNegatives: 4.0000 - FalsePositives: 7.0000 - Precision: 0.9496 - Recall: 0.9706 - TrueNegatives: 221.0000 - TruePositives: 132.0000 - accuracy: 0.9698 - loss: 0.1105 - weighted_AUC: 0.9920 - val_AUC: 0.9929 - val_FalseNegatives: 1.0000 - val_FalsePositives: 3.0000 - val_Precision: 0.9143 - val_Recall: 0.9697 - val_TrueNegatives: 55.0000 - val_TruePositives: 32.0000 - val_accuracy: 0.9560 - val_loss: 0.1033 - val_weighted_AUC: 0.9929 - learning_rate: 0.0010
Epoch 11/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - AUC: 0.9958 - FalseNegatives: 4.0000 - FalsePositives: 7.0000 - Precision: 0.9496 - Recall: 0.9706 - TrueNegatives: 221.0000 - TruePositives: 132.0000 - accuracy: 0.9698 - loss: 0.0875 - weighted_AUC: 0.9958 - val_AUC: 0.9935 - val_FalseNegatives: 1.0000 - val_FalsePositives: 3.0000 - val_Precision: 0.9143 - val_Recall: 0.9697 - val_TrueNegatives: 55.0000 - val_TruePositives: 32.0000 - val_accuracy: 0.9560 - val_loss: 0.

2026/03/04 15:16:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 15:16:08 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Successfully registered model 'RNN_Clasificacion_Best'.
Created version '1' of model 'RNN_Clasificacion_Best'.
2026/03/04 15:17:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/04 15:17:24 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Successfully registered model 'RNN_Cancer_Model'.
Created version '1' of model 'RNN_Cancer_Model'.
